In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pybfs.utilities import flow_metrics
from pybfs.calibrate_ea import _run_bfs
import math
import datetime as dt

In [ ]:
SITE = '01134500'
BASIN_AREA = 195e6  # m²
DATA_PATH = f'RV_data/calibration/{SITE}_cal.csv'

In [ ]:
# Calibrated with EA
knee_df = pd.read_csv(f'RV_data/calibration/{SITE}_knee.csv')

# Re-run BFS once with knee params to get the hydrograph
cal_df = pd.read_csv(DATA_PATH, parse_dates=['date'])
tmp_q = cal_df['discharge_m3d'].values
dys = cal_df['date'].values
streamflow_df = pd.DataFrame({'Date': pd.to_datetime(dys), 'Streamflow': tmp_q})

flow_result = flow_metrics(tmp_q, timestep='day', fr4rise=0.05)
flow = list(flow_result[:6])

row = knee_df.iloc[0]
x = np.array([
    math.log10(row['Lb']), math.log10(row['Wb']), math.log10(row['X1']),
    math.log10(row['ALPHA']), row['BETA'],
    math.log10(row['Ks']), math.log10(row['Kb']), math.log10(row['Kz']),
    math.log10(row['POR']),
])
result = _run_bfs(x, streamflow_df, flow, BASIN_AREA)
bfs_out = result[0]

# print('EA')
# display(knee_df)


# Calibrated with OG
knee_df = pd.read_csv(f'RV_data/calibration/{SITE}_orig_params.csv')

# Re-run BFS once with knee params to get the hydrograph
cal_df = pd.read_csv(DATA_PATH, parse_dates=['date'])
tmp_q = cal_df['discharge_m3d'].values
dys = cal_df['date'].values
streamflow_df = pd.DataFrame({'Date': pd.to_datetime(dys), 'Streamflow': tmp_q})

flow_result = flow_metrics(tmp_q, timestep='day', fr4rise=0.05)
flow = list(flow_result[:6])

row = knee_df.iloc[0]
x = np.array([
    math.log10(row['Lb']), math.log10(row['Wb']), math.log10(row['X1']),
    math.log10(row['ALPHA']), row['BETA'],
    math.log10(row['Ks']), math.log10(row['Kb']), math.log10(row['Kz']),
    math.log10(row['POR']),
])
result = _run_bfs(x, streamflow_df, flow, BASIN_AREA)
bfs_orig = result[0]

# print('Original')
# display(knee_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

ax.plot(bfs_out['Date'], bfs_out['Qob'], label='Observed Q', color='darkgray', lw=1)

ax.plot(bfs_out['Date'], bfs_out['Qsim'], label='Simulated Q (EA)', color='cornflowerblue', lw=0.8, linestyle='-')
ax.plot(bfs_out['Date'], bfs_out['Baseflow'], label='Baseflow (EA)', color='darkorange', lw=0.8, linestyle='--')

# ax.plot(bfs_orig['Date'], bfs_orig['Qsim'], label='Simulated Q (OG)', color='lime')
ax.plot(bfs_orig['Date'], bfs_orig['Baseflow'], label='Baseflow (OG)', color='lime', lw=1, linestyle=':')

ax.set_ylabel('Discharge (m³/day)')
ax.set_title(f'Site {SITE} — Knee-point BFS result')
ax.legend()
ax.set_ylim(0, 1e6)
ax.set_xlim(dt.datetime(2003, 11, 1), dt.datetime(2004, 5, 1))

# ax.set_yscale('log')

plt.tight_layout()

plt.show()